# Translate Experiment Instructions

For the study instructions, use Facebook model to translate the experiment to start. 

## Step 1: Extract English Content from JSON

Pull all translatable strings out of `spaml_template.json` (instructions, consent form, demographics, breaks, feedback strings) and append any that aren't already in `task_translation_template.xlsx`. Template variables like `${ window.word_key }` are preserved so translators know to leave them in place.

In [ ]:
import json
import re
import pandas as pd
from bs4 import BeautifulSoup, NavigableString
from openpyxl import load_workbook

BLOCK_TAGS  = ["h2", "h3", "p", "footer", "button", "label"]
INLINE_TAGS = ["b", "i", "a", "span"]


def _has_direct_text(tag):
    """True if tag has at least one non-whitespace direct text node."""
    return any(
        isinstance(child, NavigableString) and child.strip()
        for child in tag.children
    )


def _has_br(tag):
    return tag.find("br") is not None


def _extract_br_split(tag):
    """Split a mixed-content block at <br/> boundaries.

    Yields the <b>/<i>/etc. child text and each inter-<br> text segment
    as separate strings — mirrors how the tasks folder broke these apart.
    """
    results = []
    current = []

    for child in tag.children:
        if isinstance(child, NavigableString):
            text = child.strip()
            if text:
                current.append(text)
        elif child.name == "br":
            seg = " ".join(current)
            if seg:
                results.append(seg)
            current = []
        elif child.name in INLINE_TAGS:
            if current:
                results.append(" ".join(current))
                current = []
            inline_text = " ".join(child.get_text(separator=" ").split()).strip()
            if inline_text:
                results.append(inline_text)
        # nested block tags are handled by find_all — skip them here

    if current:
        results.append(" ".join(current))

    return results


def extract_texts_from_html(html):
    """Extract translatable strings from HTML.

    For block elements that mix an inline header (<b>Label:</b>) with body
    text separated by <br/>, split at each <br/> so header and body are
    extracted as individual strings (matching how the tasks folder stores them).

    For block elements that have plain direct text, extract at the block level.
    For block elements whose text lives entirely inside inline children, go one
    level deeper so the inline wrapper is preserved during injection.
    """
    soup = BeautifulSoup(html, "html.parser")
    texts = []
    seen = set()

    for tag in soup.find_all(BLOCK_TAGS):
        if _has_br(tag) and _has_direct_text(tag):
            # Header + body separated by <br/> — split into pieces
            for text in _extract_br_split(tag):
                text = " ".join(text.split())
                if text and text not in seen:
                    seen.add(text)
                    texts.append(text)
        elif _has_direct_text(tag):
            # Block owns its text — extract here
            text = " ".join(tag.get_text(separator=" ").split()).strip()
            if text and text not in seen:
                seen.add(text)
                texts.append(text)
        else:
            # No direct text — go into immediate inline children
            for child in tag.find_all(INLINE_TAGS, recursive=False):
                text = " ".join(child.get_text(separator=" ").split()).strip()
                if text and text not in seen:
                    seen.add(text)
                    texts.append(text)

    return texts


def extract_from_json(json_path):
    """Extract all translatable English strings from spaml_template.json."""
    with open(json_path) as f:
        data = json.load(f)

    texts = []
    seen = set()

    for comp_id, component in data["components"].items():
        for field in ["content", "context"]:
            if field not in component or not isinstance(component[field], str):
                continue
            for text in extract_texts_from_html(component[field]):
                if text not in seen:
                    seen.add(text)
                    texts.append(text)

        # JS feedback strings
        for handler in component.get("messageHandlers", []):
            code = handler.get("code", "")
            for match in re.findall(r"'(Please answer faster!|Correct!|Incorrect!)'", code):
                if match not in seen:
                    seen.add(match)
                    texts.append(match)

    return texts


def update_xlsx_template(xlsx_path, new_texts):
    """Append English strings not already in the xlsx template."""
    wb = load_workbook(xlsx_path)
    ws = wb.active

    existing = {str(row[0].value).strip() for row in ws.iter_rows() if row[0].value}

    added = 0
    for text in new_texts:
        if text not in existing:
            ws.append([text])
            added += 1

    wb.save(xlsx_path)
    return added, ws.max_row

In [ ]:
json_path = "../../03-Tasks/semantic_priming/spaml_template.json"
xlsx_path = "task_translation_template.xlsx"

new_texts = extract_from_json(json_path)
added, total = update_xlsx_template(xlsx_path, new_texts)

print(f"Extracted {len(new_texts)} strings from JSON")
print(f"Added {added} new entries to template")
print(f"Total rows in template: {total}\n")

# Show all extracted strings
display(pd.DataFrame({"English": new_texts}))

In [1]:
import os
import torch
import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoProcessor, AutoModelForSeq2SeqLM

def translate_with_backtranslation(
    template_path,
    output_base_dir,
    target_lang="ukr",
    source_lang="eng",
    batch_size=8
):
    df = pd.read_excel(template_path)
    col = df.columns[0]

    texts = df[col].astype(str).str.strip().tolist()

    forward_translations = []
    back_translations = []

    # -------------------------
    # 1) Forward translation
    # -------------------------
    for i in tqdm(range(0, len(texts), batch_size), desc="Forward translation"):
        batch = texts[i:i+batch_size]

        inputs = processor(
            text=batch,
            src_lang=source_lang,
            return_tensors="pt",
            padding=True
        ).to(device)

        with torch.no_grad():
            output_tokens = model.generate(
                **inputs,
                tgt_lang=target_lang,
                max_length=512
            )

        decoded = processor.batch_decode(
            output_tokens,
            skip_special_tokens=True
        )

        decoded = [t.strip() for t in decoded]
        forward_translations.extend(decoded)

    # -------------------------
    # 2) Back translation
    # -------------------------
    for i in tqdm(range(0, len(forward_translations), batch_size), desc="Back translation"):
        batch = forward_translations[i:i+batch_size]

        inputs = processor(
            text=batch,
            src_lang=target_lang,
            return_tensors="pt",
            padding=True
        ).to(device)

        with torch.no_grad():
            output_tokens = model.generate(
                **inputs,
                tgt_lang=source_lang,
                max_length=512
            )

        decoded = processor.batch_decode(
            output_tokens,
            skip_special_tokens=True
        )

        decoded = [t.strip() for t in decoded]
        back_translations.extend(decoded)

    # -------------------------
    # 3) Save output
    # -------------------------
    out_df = pd.DataFrame({
        "English": texts,
        "translation": forward_translations,
        "back_translation": back_translations
    })

    output_dir = os.path.join(output_base_dir, target_lang)
    os.makedirs(output_dir, exist_ok=True)

    output_path = os.path.join(output_dir, f"{target_lang}_experiment.csv")
    out_df.to_csv(output_path, index=False)

    print(f"Saved: {output_path}")

    return out_df

# Libraries and Models

In [2]:
model_name = "facebook/seamless-m4t-v2-large"
processor = AutoProcessor.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

SeamlessM4Tv2ForTextToText(
  (shared): Embedding(256102, 1024, padding_idx=0)
  (text_encoder): SeamlessM4Tv2Encoder(
    (embed_tokens): SeamlessM4Tv2ScaledWordEmbedding(256102, 1024, padding_idx=0)
    (embed_positions): SeamlessM4Tv2SinusoidalPositionalEmbedding()
    (layers): ModuleList(
      (0-23): 24 x SeamlessM4Tv2EncoderLayer(
        (self_attn): SeamlessM4Tv2Attention(
          (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
        )
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (ffn): SeamlessM4Tv2FeedForwardNetwork(
          (fc1): Linear(in_features=1024, out_features=8192, bias=True)
          (fc2): Linear(in_features=8192,

In [ ]:
# translate into ukr
translate_with_backtranslation(
    template_path="task_translation_template.xlsx",
    output_base_dir="../05_final_languages",
    target_lang="ukr"
)

## Step 2: Inject Translations into JSON

After machine translation and human review, run the cells below to inject the translated strings from `{lang}_experiment.csv` back into `spaml_template.json`, producing a language-specific `{lang}_spaml_template.json`.

**How it works:**
- Each HTML element (headings, paragraphs, footer text, labels, buttons) is matched by its normalized English text and replaced with the translation.
- Child HTML tags (e.g. `<kbd>${ window.word_key }</kbd>`) are preserved as long as their text content appears verbatim in the translation — template variables always will; translators should leave them unchanged.
- Hardcoded JS strings (`Correct!`, `Incorrect!`, `Please answer faster!`) are replaced directly in the message handler code.
- Output goes to `{lang}/{lang}_spaml_template.json` for review before uploading to the experiment platform.

In [ ]:
def rebuild_element(tag, translated_text):
    """Inject translated text into a mixed-content element, preserving child tags.

    Child tags whose text appears verbatim in the translation (e.g. template
    variables like ${ window.word_key } inside <kbd>) are kept in place.
    If a child's text isn't found it is dropped — content is still correct,
    just without that specific formatting wrapper.
    """
    if not tag.find():  # no child tags — simple replacement
        tag.string = translated_text
        return

    remaining = translated_text
    parts = []

    for child in list(tag.children):
        if not hasattr(child, "get_text"):
            continue
        child_text = child.get_text()
        if child_text in remaining:
            idx = remaining.find(child_text)
            parts.append(remaining[:idx])
            parts.append(str(child))
            remaining = remaining[idx + len(child_text):]

    parts.append(remaining)
    tag.clear()
    tag.append(BeautifulSoup("".join(parts), "html.parser"))


def _inject_br_split(tag, translation_map):
    """Replace text in a <br/>-split block tag piece by piece.

    Replaces each inline child's text and each inter-<br> text segment
    using the translation map independently, so the <b>Header:</b><br/>Body
    pattern maps correctly to the separately-translated strings.
    """
    new_parts = []

    for child in tag.children:
        if isinstance(child, NavigableString):
            text = " ".join(child.split())
            if text and text in translation_map:
                # Preserve leading/trailing whitespace from the original node
                leading = child[:len(child) - len(child.lstrip())]
                trailing = child[len(child.rstrip()):]
                new_parts.append(leading + translation_map[text] + trailing)
            else:
                new_parts.append(str(child))
        elif child.name == "br":
            new_parts.append("<br/>")
        elif child.name in INLINE_TAGS:
            inline_text = " ".join(child.get_text(separator=" ").split()).strip()
            if inline_text in translation_map:
                child_copy = BeautifulSoup(str(child), "html.parser").find()
                child_copy.string = translation_map[inline_text]
                new_parts.append(str(child_copy))
            else:
                new_parts.append(str(child))
        else:
            new_parts.append(str(child))

    tag.clear()
    tag.append(BeautifulSoup("".join(new_parts), "html.parser"))


def inject_into_json(json_path, csv_path, output_path):
    """Read a translated CSV and inject translations into a copy of the JSON template.

    Mirrors the extraction logic:
    - Block elements with <br/>-split content (e.g. <b>Header:</b><br/>Body) are
      injected piece by piece so each translated segment lands in the right place.
    - Block elements with direct plain text are replaced at the block level
      (rebuild_element preserves <kbd> wrappers for template variables).
    - Block elements whose text lives entirely inside inline children
      (e.g. <p><b>label</b></p>) are replaced at the inline child level.

    The CSV must have 'English' and 'translation' columns.
    """
    df = pd.read_csv(csv_path)
    translation_map = dict(zip(
        df["English"].str.strip(),
        df["translation"].str.strip()
    ))

    with open(json_path) as f:
        data = json.load(f)

    for comp_id, component in data["components"].items():
        for field in ["content", "context"]:
            if field not in component or not isinstance(component[field], str):
                continue

            soup = BeautifulSoup(component[field], "html.parser")

            for tag in soup.find_all(BLOCK_TAGS):
                if _has_br(tag) and _has_direct_text(tag):
                    # <b>Header:</b><br/>Body pattern — inject each piece separately
                    _inject_br_split(tag, translation_map)
                elif _has_direct_text(tag):
                    original = " ".join(tag.get_text(separator=" ").split()).strip()
                    if original in translation_map:
                        rebuild_element(tag, translation_map[original])
                else:
                    for child in tag.find_all(INLINE_TAGS, recursive=False):
                        original = " ".join(child.get_text(separator=" ").split()).strip()
                        if original in translation_map:
                            child.string = translation_map[original]

            component[field] = str(soup)

        # Replace hardcoded strings in JS message handlers
        for handler in component.get("messageHandlers", []):
            code = handler.get("code", "")
            for english, translated in translation_map.items():
                if f"'{english}'" in code:
                    code = code.replace(f"'{english}'", f"'{translated}'")
            handler["code"] = code

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"Saved translated JSON to {output_path}")

In [ ]:
target_lang = "ukr"  # change to whichever language you just translated

json_path = "../../03-Tasks/semantic_priming/spaml_template.json"
csv_path = f"../05_final_languages/{target_lang}/{target_lang}_experiment.csv"
output_path = f"../05_final_languages/{target_lang}/{target_lang}_spaml_template.json"

inject_into_json(json_path, csv_path, output_path)